In [2]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter
import yfinance as yf
from datetime import datetime
import os

# Função para calcular variação percentual
def calcular_variacao(etf_hist, data_passada, data_recente):
    if data_passada in etf_hist.index and data_recente in etf_hist.index:
        fechamento_passado = etf_hist.loc[data_passada, 'Close']
        fechamento_recente = etf_hist.loc[data_recente, 'Close']
        return ((fechamento_recente - fechamento_passado) / fechamento_passado) * 100
    else:
        return None

# Função para calcular dividend yield
def calcular_dividend_yield(etf_hist, ultimo_dia):
    ultimo_dividendo_disponivel = etf_hist['Dividends'][:ultimo_dia].loc[etf_hist['Dividends'] > 0].last_valid_index()

    if ultimo_dividendo_disponivel is not None:
        ultimo_dividendo = etf_hist.loc[ultimo_dividendo_disponivel, 'Dividends']
        fechamento_no_dia_do_dividendo = etf_hist.loc[ultimo_dividendo_disponivel, 'Close']

        # Calcula o percentual de dividendos usando o fechamento do mesmo dia do dividendo
        return (ultimo_dividendo * 12 / fechamento_no_dia_do_dividendo) * 100 if fechamento_no_dia_do_dividendo != 0 else None
    return None

# Dicionário com ETFs
etfs_list = {
    "XLK": {
        "nome": "Technology",
        "classe": "X5 - Principais",
        "ticker": "XLK"
    },
    "EWI": {
        "nome": "MSCI Italy",
        "classe": "ETFs - Regional",
        "ticker": "EWI"
    },
    "SPY": {
        "nome": "S&P 500",
        "classe": "X5 - Principais",
        "ticker": "SPY"
    },
    "EWJ": {
        "nome": "MSCI Japan",
        "classe": "ETFs - Regional",
        "ticker": "EWJ"
    },
    "EWA": {
        "nome": "MSCI Australia",
        "classe": "ETFs - Regional",
        "ticker": "EWA"
    },
    "XLF": {
        "nome": "Financials",
        "classe": "ETFs - Setores",
        "ticker": "XLF"
    },
    "GDX": {
        "nome": "Gold Miners",
        "classe": "ETFs - Commodities",
        "ticker": "GDX"
    },
}

# Criar dataframe para armazenar resultados
columns = ['Classe', 'Ticker', 'Name', 'Price', '%Day', '%Week', '%Month', '%Year', '%12m', '%DY']
results_df = pd.DataFrame(columns=columns)

# Iterar sobre cada ETF no dicionário e calcular os valores
for etf_symbol, etf_info in etfs_list.items():
    etf = yf.Ticker(etf_info['ticker'])
    etf_hist = etf.history(period="2y")

    # Pegar último dia disponível
    ultimo_dia = etf_hist.index.max()

    # Calcular variações de preço
    um_dia_atras = etf_hist.index.asof(ultimo_dia - pd.DateOffset(days=1))
    sete_dias_atras = etf_hist.index.asof(ultimo_dia - pd.tseries.offsets.BDay(7))
    trinta_dias_atras = etf_hist.index.asof(ultimo_dia - pd.tseries.offsets.BDay(30))
    doze_meses_atras = etf_hist.index.asof(ultimo_dia - pd.DateOffset(years=1))
    ano_atual = ultimo_dia.year
    primeiro_dia_ano = etf_hist[etf_hist.index.year == ano_atual].index.min()

    # Variações percentuais
    variacao_1_dia = calcular_variacao(etf_hist, um_dia_atras, ultimo_dia)
    variacao_7_dias = calcular_variacao(etf_hist, sete_dias_atras, ultimo_dia)
    variacao_30_dias = calcular_variacao(etf_hist, trinta_dias_atras, ultimo_dia)
    variacao_no_ano = calcular_variacao(etf_hist, primeiro_dia_ano, ultimo_dia)
    variacao_12_meses = calcular_variacao(etf_hist, doze_meses_atras, ultimo_dia)

    # TODO: Ajeitar função que calcula o dividend_yield
    # Preço atual e dividend yield
    preco_atual = etf_hist.loc[ultimo_dia, 'Close'] if 'Close' in etf_hist.columns else None
    dividend_yield = calcular_dividend_yield(etf_hist, ultimo_dia)

    # Criar um DataFrame temporário com os resultados
    temp_df = pd.DataFrame({
        'Classe': [etf_info['classe']],
        'Ticker': [etf_info['ticker']],
        'Name': [etf_info['nome']],
        'Price': [preco_atual],
        '%Day': [variacao_1_dia],
        '%Week': [variacao_7_dias],
        '%Month': [variacao_30_dias],
        '%Year': [variacao_no_ano],
        '%12m': [variacao_12_meses],
        '%DY': [dividend_yield]
    })

    # Concatenar o DataFrame temporário ao results_df
    results_df = pd.concat([results_df, temp_df], ignore_index=True)


def format_percentage(value):
    return f'{value:.2f}%'

# Aplicar formatação aos valores percentuais
percentage_columns = ['%Day', '%Week', '%Month', '%Year', '%12m', '%DY']
for col in percentage_columns:
    results_df[col] = results_df[col].apply(format_percentage)

# Formatar a coluna 'Price' para duas casas decimais
results_df['Price'] = results_df['Price'].apply(lambda x: f'{x:.2f}')

# Nome da pasta para os arquivos data
output_dir = 'data'
# Nome do arquivo de saída
output_file = 'data/tabelas_formatadas.xlsx'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Criar um escritor Excel
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Inicializar a linha atual
    current_row = 1
    
    # Para cada classe única no DataFrame
    for classe, group in results_df.groupby('Classe'):
        # Remover a coluna 'Classe' e resetar o índice
        table = group.drop('Classe', axis=1).reset_index(drop=True)
        
        # Escrever a tabela no Excel
        table.to_excel(writer, sheet_name='Tabelas', startrow=current_row, index=False)
        
        # Obter a planilha
        worksheet = writer.sheets['Tabelas']
        
        # Escrever o nome da classe
        worksheet.cell(row=current_row, column=1, value=classe)
        worksheet.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(table.columns))
        
        # Formatar cabeçalho da classe
        class_cell = worksheet.cell(row=current_row, column=1)
        class_cell.font = Font(bold=True, size=14)
        class_cell.fill = PatternFill(start_color="E6E6FA", end_color="E6E6FA", fill_type="solid")  # Lavanda claro
        
        # Atualizar a linha atual
        current_row += 1
        
        # Definir cores para as colunas (alternando entre duas cores)
        colors = ['FFFFE0', 'E0FFFF']  # Amarelo claro e Ciano claro
        for col_num, column in enumerate(table.columns, 1):
            fill = PatternFill(start_color=colors[col_num % 2], 
                               end_color=colors[col_num % 2], 
                               fill_type="solid")
            for row in range(current_row, current_row + len(table) + 1):
                cell = worksheet.cell(row=row, column=col_num)
                cell.fill = fill
        
        # Formatar cabeçalho das colunas
        header_fill = PatternFill(start_color="800080", end_color="800080", fill_type="solid")  # Roxo
        header_font = Font(color="FFFFFF", bold=True)  # Branco e negrito
        for col_num, column in enumerate(table.columns, 1):
            cell = worksheet.cell(row=current_row, column=col_num)
            cell.fill = header_fill
            cell.font = header_font
        
        # Atualizar a linha atual
        current_row += len(table) + 2  # +2 para adicionar uma linha em branco entre as tabelas

    # Ajustar a largura das colunas
    for column_cells in worksheet.columns:
        length = max(len(str(cell.value)) for cell in column_cells if cell.value is not None)
        worksheet.column_dimensions[get_column_letter(column_cells[0].column)].width = length + 2

print(f"As tabelas formatadas foram salvas no arquivo '{output_file}'")


C:\Users\bruno\AppData\Local\Temp\ipykernel_16796\2422289992.py:115: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, temp_df], ignore_index=True)


As tabelas formatadas foram salvas no arquivo 'data/tabelas_formatadas.xlsx'
